In [1]:
import sys
sys.path.append('../')

from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
import numpy as np
from utils import * #only needed for xgboost
import itertools
import matplotlib.pyplot as plt
from baselines import *

In [2]:
test_size_list = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95] #0.8or 0.93
target_columns = ['Volume', 'Dgv']
seed_list = [1,2,3,4,5]

In [3]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

#random_size = np.random.randint(57, 59)
#predictor_columns = np.random.choice(predictor_columns, size = random_size)
predictor_columns

['pzabovezmean',
 'pzabove2',
 'zq5',
 'zq10',
 'zq15',
 'zq20',
 'zq25',
 'zq30',
 'zq35',
 'zq40',
 'zq45',
 'zq50',
 'zq55',
 'zq60',
 'zq65',
 'zq70',
 'zq75',
 'zq80',
 'zq85',
 'zq90',
 'zq95',
 'zpcum1',
 'zpcum2',
 'zpcum3',
 'zpcum4',
 'zpcum5',
 'zpcum6',
 'zpcum7',
 'zpcum8',
 'zpcum9']

In [ ]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'val_rmse', 'val_mae', 'rmse', 'mae'])



v_list = [0.05, 0.1]
source_tree_size_list = [1,2]
target_tree_size_list = [1,2]
k_list = [0.01, 0.05]
m_0_list = [0.5, 0.9]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    source_tree_size_list,
    target_tree_size_list,
    k_list,
    m_0_list
))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:
    for test_size in test_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
            train_size = int((1-test_size)*len(data_latvia))
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
            data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

            #"General" base dataset (to use for transfer)
            X_source_train = np.array(data_sweden[predictor_columns])
            y_source_train = np.array(data_sweden[target_column])

            #Specific train and test set
            X_target_train = np.array(data_train[predictor_columns])
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns])
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns])
            y_target_test = np.array(data_test[target_column])

            print(len(X_target_train), len(X_target_val), len(X_target_test))
            for config in param_grid:
                v, source_tree_size, target_tree_size, k, m_0 = config


                #Test for all methods!!!!

                method = f'LSTransferTreeBoost'
                fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                            target_tree_size=target_tree_size, k=k, m_0=m_0)
                fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves = False)
                rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
                val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
                mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
                val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')

                ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size, method, v, source_tree_size, target_tree_size, k, m_0, val_rmse, val_mae, rmse, mae]
                ablation_transfer_real.to_csv(f'results/LSTransferTreeBoost_ablation_rs.csv')
                 
    



C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\4194774475.py:31: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


570 665 665


In [4]:
#ablation study for xgboost Gaussian errors, with gaussian source domain errors
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])


v_list = [0.01, 0.02, 0.05, 0.1, 0.15]
target_tree_size_list = [1,2,3,4]

# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    target_tree_size_list))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:
    for test_size in test_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
            train_size = int((1-test_size)*len(data_latvia))
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
            data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

            #"General" base dataset (to use for transfer)
            X_source_train = np.array(data_sweden[predictor_columns])
            y_source_train = np.array(data_sweden[target_column])

            #Specific train and test set
            X_target_train = np.array(data_train[predictor_columns])
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns])
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns])
            y_target_test = np.array(data_test[target_column])

            #additional source/target dummy sets for naive xgboost
            ones = np.ones((len(X_target_train), 1))
            X_target_train_with_dummy = np.hstack((X_target_train, ones))
            ones = np.ones((len(X_target_val), 1))
            X_target_val_with_dummy = np.hstack((X_target_val, ones))
            ones = np.ones((len(X_target_test), 1))
            X_target_test_with_dummy= np.hstack((X_target_test, ones))

            zeros = np.zeros((len(X_source_train), 1))
            X_source_train_with_dummy = np.hstack((X_source_train.copy(), zeros))

            print(len(X_target_train), len(X_target_val), len(X_target_test))
            for config in param_grid:
                v, target_tree_size = config



                #We also append baseline results!!
                method = 'xgboost'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                        
                bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds = test_xgboost(X_target_test, bst)
                val_preds = test_xgboost(X_target_val, bst)
                val_rmse = compute_rmse(val_preds, y_target_val)
                val_mae = compute_mae(val_preds, y_target_val)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size, method, v, target_tree_size, 
                                                                           val_rmse, val_mae, rmse, mae]
                
                ablation_transfer_real.to_csv(f'results/xgboost_ablation_rs.csv')

                method = 'xgboost_naive_transfer'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                X_comb = np.concatenate((X_target_train, X_source_train)) 
                y_comb = np.concatenate((y_target_train, y_source_train))       
                bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds = test_xgboost(X_target_test, bst)
                val_preds = test_xgboost(X_target_val, bst)
                val_rmse = compute_rmse(val_preds, y_target_val)
                val_mae = compute_mae(val_preds, y_target_val)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size, method, v, target_tree_size, 
                                                                           val_rmse, val_mae, rmse, mae]
                
                ablation_transfer_real.to_csv(f'results/xgboost_ablation_rs.csv')

                method = 'xgboost_naive_transfer_with_dummy'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                X_comb = np.concatenate((X_target_train_with_dummy, X_source_train_with_dummy)) 
                y_comb = np.concatenate((y_target_train, y_source_train))       
                bst = train_xgboost(X_comb, y_comb, X_target_val_with_dummy, y_target_val, boosting_rounds=1000, params=params)
                preds = test_xgboost(X_target_test_with_dummy, bst)
                val_preds = test_xgboost(X_target_val_with_dummy, bst)
                val_rmse = compute_rmse(val_preds, y_target_val)
                val_mae = compute_mae(val_preds, y_target_val)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size, method, v, target_tree_size, 
                                                                           val_rmse, val_mae, rmse, mae]
                
                ablation_transfer_real.to_csv(f'results/xgboost_ablation_rs.csv')

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\22248207.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


570 665 665


C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\22248207.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


570 665 665


C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\22248207.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


...

190 855 855


C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\22248207.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


95 902 903


C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_24032\22248207.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


95 902 903


In [ ]:
#ablation study for MLP
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 
                                                 'method', 'base_lr', 'fine_tuning_lr', 'dropout_rate', 'batch_norm', 'val_rmse', 'val_mae',
                                                   'rmse', 'mae'])

fine_tuning_lrs = [1e-4, 5e-5]
base_lrs = [5e-4, 1e-4]
dropout_list = [0.0, 0.1]
include_batch_norm = [True, False]


# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:
    for test_size in test_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
            train_size = int((1-test_size)*len(data_latvia))
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
            data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

            #"General" base dataset (to use for transfer)
            X_source_train = np.array(data_sweden[predictor_columns])
            y_source_train = np.array(data_sweden[target_column])

            #Specific train and test set
            X_target_train = np.array(data_train[predictor_columns])
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns])
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns])
            y_target_test = np.array(data_test[target_column])

            print(len(X_target_train), len(X_target_val), len(X_target_test))
                #Test for all methods!!!!

            for base_lr in base_lrs:
                for finetuning_lr in fine_tuning_lrs:
                    for dropout_rate in dropout_list:
                        for batch_norm in include_batch_norm:

                            method = f'MLP'
                            mlp = MLP(30, 100, 100, 100, 1, dropout_rate=dropout_rate, include_batch_norm=batch_norm)
                            dataloader_train = process_dataset_for_base_network(X_source_train, y_source_train)
                            mlp, train_loss, val_loss = train_mlp_on_source(dataloader_train, mlp, epochs=1000)
                            dataloader_train, dataloader_val, dataloader_test = process_datasets_for_finetuning(X_target_train, y_target_train,
                                                        X_target_val, y_target_val, X_target_test, y_target_test, batch_size=32)
                            
                            mlp, train_loss, val_loss = finetune_mlp_on_target(dataloader_train, dataloader_val, mlp, epochs=1000, freeze_layers=None)
                            rmse, mae = test_final_mlp(dataloader_test, mlp)
                            val_rmse, val_mae = test_final_mlp(dataloader_val, mlp)
                            ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size, method, base_lr, 
                                                                                       finetuning_lr, dropout_rate, batch_norm, val_rmse, val_mae, rmse, mae]
                            ablation_transfer_real.to_csv(f'results/MLP_ablation_rs.csv')


      
                        
    



C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_19800\4190362748.py:23: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


570 665 665


KeyboardInterrupt: 